# 笔记本 B

与 `notebook-A.ipynb` 配对使用。两边**故意**使用同样的变量名
（`marker`、`shared`、`count`）和同一个模块名 `probe_shared_state`，
但值不同。

只要有任何一格读到了 A 的值，就说明两个笔记本串到了同一个内核。

In [ ]:
# 与 A 同名、不同值。
marker = 'BBB'
shared = {'owner': 'B', 'items': [8, 9]}
count = 222
marker, shared, count

In [ ]:
# B 必须读到自己那份，而不是 A 的。
assert marker == 'BBB', f'marker 是 {marker!r}，可能串到了 A'
assert shared['owner'] == 'B', f'shared 属于 {shared["owner"]!r}，可能串到了 A'
assert count == 222, f'count 是 {count}，可能串到了 A'
'B 的命名空间完好'

In [ ]:
# B 也原地改动同一个可变对象。若两本共享内核，A 的断言就会失败。
shared['items'].append(-1)
shared

In [ ]:
# 与 A 用同一个模块名。若两本共用内核，这里会直接拿到 A 建好的那个
# （owner == 'A'）；各自独立时它并不存在——磁盘上没有这个 .py 文件，
# A 是在自己内核的内存里造的。
import sys
existing = sys.modules.get('probe_shared_state')
print('B 看到的模块:', '<不存在>' if existing is None else (existing.owner, existing.value))
assert existing is None or getattr(existing, 'owner', None) == 'B', \
    f'B 拿到了 {existing.owner} 的模块，两个笔记本串到了同一个内核'

In [ ]:
# B 造自己那份同名模块。若与 A 共用内核，上一格已经拿到 A 的了，
# 这里会把 owner 覆写成 'B' —— 而 A 的 `alpha-check-module` 随后就会失败。
import sys
import types
module = types.ModuleType('probe_shared_state')
module.owner = 'B'
module.value = 222
sys.modules['probe_shared_state'] = module
import probe_shared_state
(probe_shared_state.owner, probe_shared_state.value)

In [ ]:
# 关键一格：B 绝不能看见 A 的变量。预期 NameError。
# 若返回 'AAA'，两个笔记本就是共用了同一个内核。
marker_from_a

In [ ]:
# 工作目录是笔记本所在目录。两本在同一目录，所以这里相同——
# 磁盘是共享的，这不代表内核共享。
import os
os.getcwd()

In [ ]:
# 读 A 写的文件。这一格**应当成功**：磁盘是共享的，隔离指的是内核状态。
# 把它和上面几格对照看，别把两件事混为一谈。
import tempfile
from pathlib import Path
probe_file = Path(tempfile.gettempdir()) / 'dsh_probe_a_only.txt'
print('读到 A 的文件:' if probe_file.exists() else 'A 还没跑过那一格',
      probe_file.read_text(encoding='utf-8') if probe_file.exists() else '')
count

In [ ]:
# 跑它然后中断：只应打断 B。A 的内核必须不受影响。
import time
for _ in range(120):
    time.sleep(0.5)
print('不会执行到这里')

In [ ]:
# 中断后 B 自己的内核仍可用，变量仍在。
marker, count

In [ ]:
# 在 A 里按“停止内核”之后，回来跑这一格：
# B 必须还能算，证明停 A 没有连带停 B。
sum(shared['items']), marker